<a href="https://colab.research.google.com/github/VedashreeMirajkar/flyrank-machine-learning/blob/main/Copy_of_w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [ ]:

from pathlib import Path
import pandas as pd

# Find your cloned repository
REPO = Path("/content/flyrank-machine-learning")

# Check that the repository exists
if not REPO.exists():
    raise FileNotFoundError(
        "Repository not found. Run the Git clone cell first."
    )

print("Repository found!")

# Read the assignment instructions
files_to_read = [
    "README.md",
    "work/README.md",
    "DATA_USE.md",
    "skills/building-baselines/SKILL.md",
    "skills/auditing-signals/SKILL.md",
    "skills/hunting-leakage-and-validating/SKILL.md",
    "skills/flyrank/flyrank-data/SKILL.md",
]

for filename in files_to_read:
    path = REPO / filename

    print("\n" + "=" * 50)
    print("FILE:", filename)
    print("=" * 50)

    if path.exists():
        print(
            path.read_text(
                encoding="utf-8",
                errors="replace"
            )[:6000]
        )
    else:
        print("File not found.")

# Check the actual FlyRank dataset
data_path = REPO / "data/raw/content_refresh_anonymized.csv"

if not data_path.exists():
    raise FileNotFoundError(
        f"Dataset not found: {data_path}"
    )

df = pd.read_csv(data_path, nrows=5)

print("\nDATASET COLUMN NAMES:")
for column in df.columns:
    print(column)

print("\nDATA TYPES:")
print(df.dtypes.to_string())

print("\nDataset preview:")
print(df.head(3).to_string(index=False))


Repository found!

FILE: README.md
# FlyRank ML Internship — Starter Repo

**Applied Search Intelligence: Google Search Ranking & Discoverability**

This is the starting point for the FlyRank ML Internship. You **clone it into your own public
repo** (one click — *Use this template*), build everything there, and submit that repo URL on
each assignment in your portal — it's your workspace, your submission, and your portfolio all
at once. The rhythm is simple: do the work, commit it, submit on the card. Done.

Everything here runs on a small **anonymized** slice of real FlyRank search data. No credentials,
no private client data, no setup headaches.

> **New here?** Two reads: **[SETUP.md](SETUP.md)** (GitHub, Colab, and data access — ten
> minutes, with every silent pitfall flagged), then **[GUIDE.md](GUIDE.md)** (every file
> explained, what to edit vs. leave alone, and where your own work goes — five minutes).

---

## Quickstart — first win in 2 minutes

The fastest path is Google Col

In [ ]:

from pathlib import Path
import pandas as pd

REPO = Path("/content/flyrank-machine-learning")
DATA = REPO / "data/raw/content_refresh_anonymized.csv"

# Load the actual FlyRank dataset
df = pd.read_csv(DATA)

# Display a short summary
print("SUCCESS: FlyRank dataset loaded")
print("Rows:", len(df))
print("Columns:", len(df.columns))

print("\nCOLUMN NAMES:")
print(df.columns.tolist())

print("\nDATA TYPES:")
print(df.dtypes.to_string())

print("\nMISSING VALUES:")
print(df.isnull().sum().to_string())

# Display a few rows for checking
display(df.head(3))


SUCCESS: FlyRank dataset loaded
Rows: 30000
Columns: 44

COLUMN NAMES:
['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']

DATA TYPES:
content_id                 object
client_id                  object
search_volume             float64
competition               float64
competiti

,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9


## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

### 1. My rule and its reason codes

**Baseline rule**

This baseline ranks candidate actions using only the eligible, observed signals available in the dataset. It prioritizes actions supported by stronger measured evidence, while considering signal freshness, volume, and the reliability of the available observations where these fields exist. The score is a directional decision-support score, not a prediction of guaranteed outcomes.

The rule must follow the repository's documented data definitions, eligibility constraints, and permitted observation windows. It must not use product flags, future-window information, client names, URLs, or private queries as scoring inputs.

**Reason codes**

- `FRESH_SIGNAL`: The action is supported by a relatively fresh observed signal.
- `HIGH_VOLUME`: The action has comparatively strong observed volume.
- `CTR_POSITION_SIGNAL`: The observed click-through-rate and position information supports further investigation.
- `QUICK_WIN`: The action meets the repository-defined criteria for a potential quick win.
- `MIXED_EVIDENCE`: Available signals disagree or provide incomplete support.
- `INSUFFICIENT_EVIDENCE`: The available observations do not support a confident recommendation.

Reason codes will be assigned only when the relevant fields and repository-defined conditions support them. These labels describe observed evidence, not proven causation.

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [ ]:

from pathlib import Path
import pandas as pd
import numpy as np

# 1. Locate the repository and dataset
ROOT = Path("/content/flyrank-machine-learning")
DATA_PATH = ROOT / "data/raw/content_refresh_anonymized.csv"
OUTPUT_DIR = ROOT / "work/outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(DATA_PATH)

# 2. Validate the columns required by the baseline
required_columns = [
    "content_id",
    "content_age_days",
    "days_since_last_update",
]

missing_columns = [c for c in required_columns if c not in df.columns]
if missing_columns:
    raise ValueError(f"Missing required columns: {missing_columns}")

# 3. Prepare numeric signals
age = pd.to_numeric(df["content_age_days"], errors="coerce")
days_since_update = pd.to_numeric(
    df["days_since_last_update"], errors="coerce"
)

# Missing or negative values are not treated as evidence of staleness.
age_valid = age.notna() & age.ge(0)
update_valid = days_since_update.notna() & days_since_update.ge(0)

old_content = age_valid & age.ge(180)
stale_content = update_valid & days_since_update.ge(180)

# 4. Apply the baseline score
# Higher scores mean stronger evidence for reviewing content for refresh.
baseline_score = (
    stale_content.astype(int) * 2
    + old_content.astype(int)
)

# 5. Assign repository-specific reason codes
reason_code = np.select(
    [
        stale_content & old_content,
        stale_content & ~old_content,
        ~stale_content & old_content,
    ],
    [
        "stale_and_old",
        "stale_content",
        "old_but_recently_updated",
    ],
    default="not_stale_or_old",
)

# 6. Create the ranked action queue
queue = pd.DataFrame({
    "content_id": df["content_id"],
    "action": np.where(
        stale_content,
        "review_for_refresh",
        "monitor_or_review_manually",
    ),
    "baseline_score": baseline_score,
    "reason_code": reason_code,
    "content_age_days": age,
    "days_since_last_update": days_since_update,
})

# Missing signals reduce interpretability; do not silently treat them as fresh.
queue["confidence_note"] = np.select(
    [
        ~age_valid & ~update_valid,
        ~age_valid | ~update_valid,
    ],
    [
        "Low: both age and update signals unavailable or invalid",
        "Limited: one age or update signal unavailable or invalid",
    ],
    default="Rule-based confidence only; validate before acting",
)

# Stable ordering: score descending, then content ID ascending.
queue = queue.sort_values(
    ["baseline_score", "content_id"],
    ascending=[False, True],
    kind="mergesort",
).reset_index(drop=True)

queue.insert(0, "baseline_rank", range(1, len(queue) + 1))

# 7. Write the required CSV
output_path = OUTPUT_DIR / "baseline_action_score.csv"
queue.to_csv(output_path, index=False)

print("Baseline scoring completed.")
print("Rows ranked:", len(queue))
print("Output:", output_path)
print("\nReason-code counts:")
print(queue["reason_code"].value_counts(dropna=False))
print("\nTop 10 ranked actions:")
display(queue.head(10))


Baseline scoring completed.
Rows ranked: 30000
Output: /content/flyrank-machine-learning/work/outputs/baseline_action_score.csv

Reason-code counts:
reason_code
old_but_recently_updated    17812
not_stale_or_old            12014
stale_and_old                 174
Name: count, dtype: int64

Top 10 ranked actions:


,baseline_rank,content_id,action,baseline_score,reason_code,content_age_days,days_since_last_update,confidence_note
0,1,content_0173fb0dc986,review_for_refresh,3,stale_and_old,232,183,Rule-based confidence only; validate before ac...
1,2,content_026a1e2a82fd,review_for_refresh,3,stale_and_old,313,305,Rule-based confidence only; validate before ac...
2,3,content_02b0d6e30129,review_for_refresh,3,stale_and_old,313,313,Rule-based confidence only; validate before ac...
3,4,content_06e19c6486b0,review_for_refresh,3,stale_and_old,334,334,Rule-based confidence only; validate before ac...
4,5,content_06f581dd9383,review_for_refresh,3,stale_and_old,302,211,Rule-based confidence only; validate before ac...
5,6,content_074ba6ead17b,review_for_refresh,3,stale_and_old,232,183,Rule-based confidence only; validate before ac...
6,7,content_07ce98c6085a,review_for_refresh,3,stale_and_old,305,304,Rule-based confidence only; validate before ac...
7,8,content_0943559d5506,review_for_refresh,3,stale_and_old,305,211,Rule-based confidence only; validate before ac...
8,9,content_09af0f22fdc4,review_for_refresh,3,stale_and_old,313,211,Rule-based confidence only; validate before ac...
9,10,content_0a91db491d14,review_for_refresh,3,stale_and_old,231,193,Rule-based confidence only; validate before ac...


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [ ]:

# ML-07 — Section 3: Top-20 review

from pathlib import Path
import pandas as pd

ROOT = Path("/content/flyrank-machine-learning")
INPUT_PATH = ROOT / "work/outputs/baseline_action_score.csv"
OUTPUT_PATH = ROOT / "work/outputs/top20_review.csv"

queue = pd.read_csv(INPUT_PATH)
top20 = queue.head(20).copy()

def explain_reason(row):
    reason = row["reason_code"]

    explanations = {
        "stale_and_old": (
            "Both content age and time since last update meet "
            "the 180-day threshold."
        ),
        "stale_content": (
            "Time since last update meets the 180-day threshold."
        ),
        "old_but_recently_updated": (
            "Content is old, but the last-update signal does not "
            "meet the staleness threshold."
        ),
        "not_stale_or_old": (
            "Neither available signal meets the baseline threshold."
        ),
    }
    return explanations.get(
        reason, "Review the available signals and rule conditions."
    )

def confidence_note(row):
    age_missing = pd.isna(row["content_age_days"])
    update_missing = pd.isna(row["days_since_last_update"])

    if age_missing and update_missing:
        return "Low: both signals are missing."
    if age_missing or update_missing:
        return "Limited: one signal is missing."
    return (
        "Moderate for applying this simple rule; "
        "not a measure of outcome probability."
    )

def what_could_make_it_wrong(row):
    reason = row["reason_code"]

    if reason == "stale_and_old":
        return (
            "The content may still be accurate and useful; "
            "age alone does not prove a refresh is needed."
        )
    if reason == "stale_content":
        return (
            "The content may not need updating, or the recorded "
            "update date may not reflect meaningful changes."
        )
    if reason == "old_but_recently_updated":
        return (
            "A recent update may have been minor, or the content "
            "may still need substantive review."
        )
    return (
        "The content may need attention for reasons this "
        "two-signal baseline does not measure."
    )

top20["reason_explanation"] = top20.apply(explain_reason, axis=1)
top20["confidence_note"] = top20.apply(confidence_note, axis=1)
top20["what_could_make_it_wrong"] = top20.apply(
    what_could_make_it_wrong, axis=1
)

review_columns = [
    "baseline_rank",
    "content_id",
    "action",
    "reason_code",
    "reason_explanation",
    "confidence_note",
    "what_could_make_it_wrong",
]

top20_review = top20[review_columns]
top20_review.to_csv(OUTPUT_PATH, index=False)

print("Top-20 review created:", OUTPUT_PATH)
display(top20_review)


Top-20 review created: /content/flyrank-machine-learning/work/outputs/top20_review.csv


,baseline_rank,content_id,action,reason_code,reason_explanation,confidence_note,what_could_make_it_wrong
0,1,content_0173fb0dc986,review_for_refresh,stale_and_old,Both content age and time since last update me...,Moderate for applying this simple rule; not a ...,The content may still be accurate and useful; ...
1,2,content_026a1e2a82fd,review_for_refresh,stale_and_old,Both content age and time since last update me...,Moderate for applying this simple rule; not a ...,The content may still be accurate and useful; ...
2,3,content_02b0d6e30129,review_for_refresh,stale_and_old,Both content age and time since last update me...,Moderate for applying this simple rule; not a ...,The content may still be accurate and useful; ...
3,4,content_06e19c6486b0,review_for_refresh,stale_and_old,Both content age and time since last update me...,Moderate for applying this simple rule; not a ...,The content may still be accurate and useful; ...
4,5,content_06f581dd9383,review_for_refresh,stale_and_old,Both content age and time since last update me...,Moderate for applying this simple rule; not a ...,The content may still be accurate and useful; ...
5,6,content_074ba6ead17b,review_for_refresh,stale_and_old,Both content age and time since last update me...,Moderate for applying this simple rule; not a ...,The content may still be accurate and useful; ...
6,7,content_07ce98c6085a,review_for_refresh,stale_and_old,Both content age and time since last update me...,Moderate for applying this simple rule; not a ...,The content may still be accurate and useful; ...
7,8,content_0943559d5506,review_for_refresh,stale_and_old,Both content age and time since last update me...,Moderate for applying this simple rule; not a ...,The content may still be accurate and useful; ...
8,9,content_09af0f22fdc4,review_for_refresh,stale_and_old,Both content age and time since last update me...,Moderate for applying this simple rule; not a ...,The content may still be accurate and useful; ...
9,10,content_0a91db491d14,review_for_refresh,stale_and_old,Both content age and time since last update me...,Moderate for applying this simple rule; not a ...,The content may still be accurate and useful; ...


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [ ]:

# ML-07 — Section 4: Weak picks and leakage check

import pandas as pd
from pathlib import Path

ROOT = Path("/content/flyrank-machine-learning")
DATA_PATH = ROOT / "data/raw/content_refresh_anonymized.csv"
QUEUE_PATH = ROOT / "work/outputs/baseline_action_score.csv"

source = pd.read_csv(DATA_PATH)
queue = pd.read_csv(QUEUE_PATH)

print("Source rows:", len(source))
print("Queue rows:", len(queue))
print("Duplicate content IDs:", queue["content_id"].duplicated().sum())

# Inspect the strongest picks and potential weaknesses.
print("\nTop-20 reason-code distribution:")
print(queue.head(20)["reason_code"].value_counts())

print("\nPotentially weak picks: old content without a stale update signal")
weak_picks = queue[
    queue["reason_code"] == "old_but_recently_updated"
].head(10)

display(weak_picks)

# Inspect sensitive or potentially future-oriented field names.
suspicious_columns = [
    col for col in source.columns
    if any(term in col.lower() for term in [
        "product_flag", "future", "conversion_after",
        "outcome_after", "label", "target"
    ])
]

print("\nColumns requiring leakage review:")
print(suspicious_columns if suspicious_columns else "None detected by name")

# Check whether scoring inputs appear in the source and queue.
score_inputs = ["content_age_days", "days_since_last_update"]
print("\nScoring input availability:")
for col in score_inputs:
    print(f"{col}: source={col in source.columns}, queue={col in queue.columns}")

print(
    "\nCAUTION: This is a preliminary name-based check, not proof "
    "that the notebook is leakage-free. Inspect DATA_USE.md and "
    "the repository instructions to verify field definitions, "
    "allowed observation windows, and prohibited product flags."
)


Source rows: 30000
Queue rows: 30000
Duplicate content IDs: 0

Top-20 reason-code distribution:
reason_code
stale_and_old    20
Name: count, dtype: int64

Potentially weak picks: old content without a stale update signal


,baseline_rank,content_id,action,baseline_score,reason_code,content_age_days,days_since_last_update,confidence_note
174,175,content_0003a66b9fa5,monitor_or_review_manually,1,old_but_recently_updated,329,104,Rule-based confidence only; validate before ac...
175,176,content_00042377f909,monitor_or_review_manually,1,old_but_recently_updated,256,104,Rule-based confidence only; validate before ac...
176,177,content_0008d03a5c6c,monitor_or_review_manually,1,old_but_recently_updated,250,20,Rule-based confidence only; validate before ac...
177,178,content_000b07fd81c1,monitor_or_review_manually,1,old_but_recently_updated,441,22,Rule-based confidence only; validate before ac...
178,179,content_000c896649cc,monitor_or_review_manually,1,old_but_recently_updated,310,20,Rule-based confidence only; validate before ac...
179,180,content_0011472f3d4e,monitor_or_review_manually,1,old_but_recently_updated,230,20,Rule-based confidence only; validate before ac...
180,181,content_00144b1e6482,monitor_or_review_manually,1,old_but_recently_updated,300,104,Rule-based confidence only; validate before ac...
181,182,content_0015d80e7092,monitor_or_review_manually,1,old_but_recently_updated,291,20,Rule-based confidence only; validate before ac...
182,183,content_001ae9124afc,monitor_or_review_manually,1,old_but_recently_updated,307,104,Rule-based confidence only; validate before ac...
183,184,content_001b45a25ada,monitor_or_review_manually,1,old_but_recently_updated,330,104,Rule-based confidence only; validate before ac...



Columns requiring leakage review:
None detected by name

Scoring input availability:
content_age_days: source=True, queue=True
days_since_last_update: source=True, queue=True

CAUTION: This is a preliminary name-based check, not proof that the notebook is leakage-free. Inspect DATA_USE.md and the repository instructions to verify field definitions, allowed observation windows, and prohibited product flags.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.